# Satellite GeoTIFF Viewer

Interactive notebook for inspecting any satellite GeoTIFF. Upload or point to a `.tif`, then explore RGB composites, individual bands, threshold overlays, histograms, and metadata without loading the full raster into memory.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    %pip install -q rasterio matplotlib numpy pandas ipywidgets


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/IGCD')
except Exception:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
UPLOAD_DIR = PROJECT_ROOT / 'data' / 'viewer_uploads'
PREVIEW_DIR = PROJECT_ROOT / 'reports' / 'satellite_viewer_previews'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
print({'project_root': str(PROJECT_ROOT), 'upload_dir': str(UPLOAD_DIR)})


In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

from igcd.raster_viewer import (
    band_statistics,
    default_rgb_bands,
    open_raster_preview,
    overlay_mask,
    rgb_composite,
    single_band_display,
    threshold_mask,
)


In [ ]:
# Option A: upload a GeoTIFF in Colab.
# Option B: set TIF_PATH manually to an existing file in Drive.

TIF_PATH = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded))
        uploaded_path = UPLOAD_DIR / first_name
        uploaded_path.write_bytes(uploaded[first_name])
        TIF_PATH = uploaded_path
        print(f'Uploaded: {TIF_PATH}')
except Exception:
    print('Upload widget is available only in Colab. Set TIF_PATH manually below.')

# Example manual path:
# TIF_PATH = PROJECT_ROOT / 'exports' / '2018' / 'IGCD_000001_2018_sentinel.tif'


In [ ]:
# If you did not upload above, set this manually and run this cell.
# TIF_PATH = Path('/content/drive/MyDrive/IGCD_exports/IGCD_000001_2018_sentinel.tif')

if TIF_PATH is None:
    raise ValueError('Set TIF_PATH to a GeoTIFF path or upload a file first.')

MAX_PREVIEW_SIZE = 1400
preview = open_raster_preview(TIF_PATH, max_size=MAX_PREVIEW_SIZE)
stats_df = pd.DataFrame(band_statistics(preview))
print({'file': str(preview.path), 'bands': preview.band_count, **preview.metadata})
display(stats_df)


In [ ]:
band_options = [(label, idx) for idx, label in enumerate(preview.band_labels, start=1)]
default_r, default_g, default_b = default_rgb_bands(preview.band_count)

mode = widgets.ToggleButtons(
    options=[('RGB composite', 'rgb'), ('Single band', 'single'), ('Overlay', 'overlay')],
    value='rgb',
    description='View',
)
red = widgets.Dropdown(options=band_options, value=default_r, description='Red')
green = widgets.Dropdown(options=band_options, value=default_g, description='Green')
blue = widgets.Dropdown(options=band_options, value=default_b, description='Blue')
single = widgets.Dropdown(options=band_options, value=1, description='Band')
overlay_band = widgets.Dropdown(options=band_options, value=1, description='Overlay')
stretch = widgets.Dropdown(
    options=[('Percentile 2-98', 'percentile'), ('Min / max', 'minmax'), ('Mean +/- 2 std', 'stddev')],
    value='percentile',
    description='Stretch',
)
low = widgets.FloatSlider(value=2, min=0, max=20, step=0.5, description='Low %')
high = widgets.FloatSlider(value=98, min=80, max=100, step=0.5, description='High %')
gamma = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='Gamma')
colormap = widgets.Dropdown(
    options=['gray', 'viridis', 'plasma', 'magma', 'terrain', 'turbo', 'Blues'],
    value='gray',
    description='Cmap',
)
threshold = widgets.FloatText(value=float(np.nanmean(preview.data[0])), description='Threshold')
direction = widgets.Dropdown(
    options=[('>= threshold', 'greater_equal'), ('<= threshold', 'less_equal')],
    value='greater_equal',
    description='Rule',
)
direction_labels = {value: label for label, value in direction.options}
opacity = widgets.FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05, description='Opacity')
show_hist = widgets.Checkbox(value=True, description='Histogram')
save_button = widgets.Button(description='Save PNG', button_style='')
status = widgets.HTML(value='')
out = widgets.Output()

last_image = {'array': None, 'mode': None}


def _plot_hist(ax, values):
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        ax.text(0.5, 0.5, 'No valid pixels', ha='center', va='center')
        return
    ax.hist(finite.ravel(), bins=80, color='0.35')
    ax.set_title('Preview histogram')
    ax.set_xlabel('Pixel value')
    ax.set_ylabel('Count')


def render(*_):
    with out:
        clear_output(wait=True)
        fig_cols = 2 if show_hist.value else 1
        fig, axes = plt.subplots(1, fig_cols, figsize=(8 * fig_cols, 8))
        if fig_cols == 1:
            axes = [axes]
        ax = axes[0]
        if mode.value == 'rgb':
            image = rgb_composite(
                preview,
                red.value,
                green.value,
                blue.value,
                mode=stretch.value,
                lower_percentile=low.value,
                upper_percentile=high.value,
                gamma=gamma.value,
            )
            ax.imshow(image)
            ax.set_title(f'RGB: R{red.value} G{green.value} B{blue.value}')
            hist_values = preview.data[red.value - 1]
        elif mode.value == 'single':
            image = single_band_display(
                preview,
                single.value,
                mode=stretch.value,
                lower_percentile=low.value,
                upper_percentile=high.value,
                gamma=gamma.value,
            )
            ax.imshow(image, cmap=colormap.value)
            ax.set_title(f'Single band: {single.value}')
            hist_values = preview.data[single.value - 1]
        else:
            base = rgb_composite(
                preview,
                red.value,
                green.value,
                blue.value,
                mode=stretch.value,
                lower_percentile=low.value,
                upper_percentile=high.value,
                gamma=gamma.value,
            )
            mask = threshold_mask(
                preview,
                overlay_band.value,
                threshold.value,
                direction=direction.value,
            )
            image = overlay_mask(base, mask, alpha=opacity.value)
            ax.imshow(image)
            ax.set_title(
                f'Overlay band {overlay_band.value}: '
                f'{direction_labels[direction.value]} {threshold.value:g}'
            )
            hist_values = preview.data[overlay_band.value - 1]
        ax.axis('off')
        last_image['array'] = image
        last_image['mode'] = mode.value
        if show_hist.value:
            _plot_hist(axes[1], hist_values)
        fig.tight_layout()
        plt.show()


def save_png(_):
    if last_image['array'] is None:
        render()
    safe_name = preview.path.stem.replace(' ', '_')
    output = PREVIEW_DIR / f'{safe_name}_{last_image["mode"]}_preview.png'
    plt.imsave(output, last_image['array'], cmap=None if last_image['array'].ndim == 3 else colormap.value)
    status.value = f'<b>Saved:</b> {output}'

for widget in [mode, red, green, blue, single, overlay_band, stretch, low, high, gamma, colormap, threshold, direction, opacity, show_hist]:
    widget.observe(render, names='value')
save_button.on_click(save_png)

controls = widgets.VBox([
    mode,
    widgets.HBox([red, green, blue]),
    widgets.HBox([single, colormap]),
    widgets.HBox([overlay_band, threshold, direction, opacity]),
    widgets.HBox([stretch, low, high, gamma]),
    widgets.HBox([show_hist, save_button]),
    status,
])

display(controls, out)
render()


In [ ]:
# Optional: show metadata in a cleaner table.
metadata_df = pd.DataFrame(
    [{'property': key, 'value': value} for key, value in preview.metadata.items()]
)
display(metadata_df)
